In [ ]:
"""
Environment Setup Module.
This cell installs the necessary quantum computing libraries in the Google Colab environment.
It must be executed before running any other cell.
"""
!pip install qiskit[visualization] qiskit-aer matplotlib pylatexenc

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE! You can now proceed to the next cell.")

### Exercise 3: Deutsch's Algorithm and the Quantum Oracle

**Motivation:**
Classically, if you have a black-box function $f(x)$ taking a 1-bit input, you must evaluate it twice ($x=0$ and $x=1$) to determine if the output is always the same (**Constant**) or divided 50/50 (**Balanced**). Deutsch's algorithm solves this in exactly 1 query by exploiting **Quantum Interference** and **Phase Kickback**.

**Theoretical Context:**
The algorithm uses an input qubit and an ancilla (workspace) qubit. By preparing the ancilla in the $|-\rangle$ state, the Oracle's output is not written to the target's probabilities, but instead "kicked back" as a phase shift on the input qubit. A final Hadamard gate translates this phase difference back into a measurable 0 or 1.

**Your Task:**
1. Run the code below exactly as provided. The Oracle section contains a `CNOT` gate, which acts as the balanced function $f(x) = x$. 
2. **Observe the result:** A 100% probability of measuring `1` proves the function is Balanced.
3. **The Experiment:** Go to the `STUDENT EXPERIMENT ZONE` in the code. Comment out the `qc.cx(0, 1)` line by typing a `#` in front of it. This removes the gate entirely, making the Oracle act as an Identity gate ($f(x) = 0$, which is a Constant function).
4. Run the code again. **Observe:** The interference perfectly flips, yielding a 100% probability of measuring `0`.

In [ ]:
"""
Deutsch's Algorithm Module.
Demonstrates quantum advantage by determining the global property 
(constant vs. balanced) of an oracle function in a single query.
"""

def run_deutsch_algorithm() -> None:
    """
    Builds, executes, and visualizes Deutsch's algorithm.
    The function evaluates an oracle and prints the logical conclusion.
    """
    # Initialize: 1 Input Qubit (q0), 1 Ancilla Qubit (q1), 1 Classical Bit
    qc = QuantumCircuit(2, 1)

    # --- 1. STATE PREPARATION ---
    # Prepare ancilla in |1> before applying the Hadamard gate
    qc.x(1)           
    
    # Apply Hadamard to both to create superposition and enable phase kickback
    qc.h(0)           # Input to |+>
    qc.h(1)           # Ancilla to |->
    qc.barrier()      # Visual and optimization barrier

    # --- 2. THE ORACLE (BLACK BOX) ---
    # ---------------------------------------------------------
    # 🧪 STUDENT EXPERIMENT ZONE: 
    # The CNOT gate below makes the oracle BALANCED (f(x) = x).
    # To make it CONSTANT (f(x) = 0), add a '#' at the start of the next line.
    
    qc.cx(0, 1)
    
    # ---------------------------------------------------------
    qc.barrier()

    # --- 3. INTERFERENCE & MEASUREMENT ---
    # Apply final Hadamard to the input qubit to extract the relative phase
    qc.h(0)           
    
    # Measure the input qubit (q0) into the classical bit (c0)
    qc.measure(0, 0)  

    # Draw the circuit diagram
    display(qc.draw('mpl'))

    # --- 4. EXECUTION ---
    # Run the circuit on the local simulator
    simulator = AerSimulator()
    job = simulator.run(qc, shots=1000)
    counts = job.result().get_counts()

    # --- 5. ANALYSIS ---
    print(f"Measurement Results: {counts}")
    
    # Interpret the results based on Deutsch's mathematical proof
    if '1' in counts and counts.get('1', 0) > 500:
        print("🎯 Conclusion: The function is BALANCED (f(0) != f(1))")
    else:
        print("🎯 Conclusion: The function is CONSTANT (f(0) == f(1))")
        
    display(plot_histogram(counts))

# Execute the algorithm
run_deutsch_algorithm()